# 🧬 Sturm Pipeline — Manifest Builder v2.1
## Parser corregido: dual-format A/B + normalización de treatments

### Cambios vs v2.0
| Problema | v2.0 | v2.1 |
|---|---|---|
| Nombres con `_` (Formato B) | ❌ magnif y passage fallaban | ✅ parser dedicado |
| `seahorse well, 10x` → treatment=`10x` | ❌ falso positivo | ✅ filtrado |
| `3days growth, post-thaw, 10x` | ❌ treatment ruidoso | ✅ limpio |
| `HEK293` en carpetas | ❌ cell_line=None sin flag | ✅ marcado como SKIP |
| `puls` vs `Puls` vs `a-Keto` vs `a-keto` | ❌ tratados como distintos | ✅ canónico único |
| cell_line sin `-m`/`-f` (`hFB14` vs `hFB14-m`) | ❌ no matcheaban en join | ✅ normalización numérica |

**Resultado esperado:** de ~1188 imágenes con parsing incompleto → <400

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import re
import json
import hashlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from datetime import datetime
from collections import defaultdict

# ─────────────────────────────────────────────────────────────────────────────
# ⚙️  RUTAS — igual que antes
# ─────────────────────────────────────────────────────────────────────────────
CSV_PATH    = Path(r'/Users/JCB/Documentos/Proyecto Integrador/data/Lifespan_Study_Data.csv')
IMAGES_ROOT = Path(r'/Volumes/SanDisk SSD 1TB/Storage/Data/Cellular_Lifespan_Study_Brightfield_Images')
OUTPUT_DIR  = Path(r'/Users/JCB/Documentos/Proyecto Integrador/data/manifests')
# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'logs').mkdir(exist_ok=True)
(OUTPUT_DIR / 'reports').mkdir(exist_ok=True)

PIPELINE_VERSION = '2.1.0'
RUN_DATE = datetime.now().strftime('%Y-%m-%d')
print(f'✅ Setup OK — versión {PIPELINE_VERSION} | {RUN_DATE}')

✅ Setup OK — versión 2.1.0 | 2026-03-08


## 2. Cargar y estandarizar CSV (sin cambios)

In [2]:
COLUMN_MAP = {
    'Sample': 'sample_id', 'Cell_Line': 'cell_line', 'Cell_line_new': 'cell_line_clean',
    'Cell_Line_Group': 'cell_line_group', 'Unique_Variable_Name': 'unique_var_name',
    'Replicate_Line': 'replicate_line', 'ShinyApp_Replicate_Line': 'shiny_replicate_line',
    'Study_Part': 'study_part', 'Cell_Type': 'cell_type', 'Sex': 'sex',
    'Clinical_Condition': 'clinical_condition', 'Donor_Age': 'donor_age',
    'Passage': 'passage', 'Time_Point': 'timepoint',
    'Population_Doublings_DI': 'pdl', 'MiAge_Population_Doublings': 'pdl_miage',
    'Percent_Lifespan': 'pct_lifespan', 'Days_Grown': 'days_grown',
    'Date_of_Passage': 'date_passage', 'growth_rate': 'growth_rate',
    'Divisions_per_Passage_DI': 'divisions_per_passage',
    'Treatment': 'treatment', 'Treatments': 'treatments_detail',
    'Treatment_description': 'treatment_desc', 'Percent_Oxygen': 'pct_oxygen',
    'Telomere_Length': 'telomere_length', 'Copy_Number': 'mtdna_cn',
    'cf_mtDNA': 'cf_mtdna', 'cf_mtDNA_per_nDNA': 'cf_mtdna_ratio',
    'Cell_Size': 'cell_size', 'Percent_Dead': 'pct_dead',
    'IL6': 'il6', 'GDF15': 'gdf15',
    'Horvath1': 'clock_horvath1', 'PhenoAge': 'clock_phenoage',
    'GrimAge': 'clock_grimage', 'Mitotic_Age': 'clock_mitotic',
    'DNAmTL': 'clock_dnam_telomere', 'DNAmSen': 'clock_dnam_sen',
    'DunedinPoAm_38': 'clock_dunedin',
    'Basal_Respiration': 'seahorse_basal', 'Max_Respiration': 'seahorse_max',
    'ATP_Linked_Respiration': 'seahorse_atp', 'Coupling_Efficiency': 'seahorse_coupling',
    'ATPratio': 'atp_ratio',
    'RNAseq_ID': 'rnaseq_id', 'basename': 'methylation_basename',
    'TechRep': 'tech_rep', 'freeze_thaw_count': 'freeze_thaw_count',
    'pre_study_passages': 'pre_study_passages',
}

df_raw = pd.read_csv(CSV_PATH, low_memory=False)
df = df_raw.rename(columns={k: v for k, v in COLUMN_MAP.items() if k in df_raw.columns}).copy()
print(f'CSV: {df.shape[0]} filas × {df.shape[1]} columnas')

df['sample_id'] = df['sample_id'].astype(str).str.strip()
df['pdl'] = pd.to_numeric(df['pdl'], errors='coerce')

# PDL normalizado por cell_line
group_col = 'cell_line' if 'cell_line' in df.columns else 'sample_id'
df['pdl_norm'] = df.groupby(group_col)['pdl'].transform(
    lambda x: (x - x.min()) / (x.max() - x.min() + 1e-8)
)
df['pdl_bin'] = pd.cut(df['pdl_norm'], bins=[0, 0.33, 0.67, 1.01],
                        labels=['early', 'mid', 'late'], include_lowest=True)

print(f'PDL rango: {df["pdl"].min():.1f} → {df["pdl"].max():.1f}')
print(f'Distribución pdl_bin: {df["pdl_bin"].value_counts().to_dict()}')
print(f'Condición clínica: {df["clinical_condition"].value_counts().to_dict()}')

CSV: 1919 filas × 271 columnas
PDL rango: 0.0 → 346.7
Distribución pdl_bin: {'late': 667, 'mid': 639, 'early': 594}
Condición clínica: {'Normal': 1666, 'SURF1_Mutation': 253}


## 3. Parser v2.1 — Dual-format con detección automática

### Los 3 formatos reales encontrados:
```
Formato A:  hFB1-m P10  t175 4 days growth 10x, Ctrl       ← separador=espacio, treatment tras coma
Formato B:  hFB14_Acute_DEX_P27_7-days_250kcells_10x       ← separador=guión_bajo
Formato C:  ctrl, P10, 500kcells, t175, 5-days growth, 10x ← sin cell_line, cell_line viene de carpeta
SKIP:       HEK293, PXX, 250Kcells, t75, 7-days growth     ← no es fibroblasto
```

In [3]:
# ── Tabla canónica de treatments ─────────────────────────────────────────────
# Todas las variantes conocidas → nombre canónico
TREATMENT_CANON = {
    # Control
    'ctrl': 'Control', 'control': 'Control',
    # DEX (glucocorticoid)
    'dex': 'DEX', 'glucocorticoid': 'DEX',
    # MitoQ
    'mitoq': 'MitoQ', 'mito-q': 'MitoQ', 'mito q': 'MitoQ',
    # NAC
    'nac': 'NAC',
    # Combinaciones
    'mitoq+dex': 'MitoQ+DEX', 'mitoq + dex': 'MitoQ+DEX', 'mito-q+dex': 'MitoQ+DEX',
    'nac+dex': 'NAC+DEX', 'nac + dex': 'NAC+DEX',
    # Vehicle
    'vehicle': 'Vehicle',
    # Pulso
    'puls': 'Puls', 'pulse': 'Puls',
    # Alpha-Ketoglutarato
    'a-keto': 'Alpha-Keto', 'a-ketoglutarate': 'Alpha-Keto',
    'alpha-keto': 'Alpha-Keto', 'alpha-ketoglutarate': 'Alpha-Keto',
    'a-kg': 'Alpha-Keto', 'akg': 'Alpha-Keto',
    # Seahorse drugs
    'oligomycin': 'Oligomycin', 'oligo': 'Oligomycin',
    'rotenone': 'Rotenone',
    'fccp': 'FCCP',
    'antimycin': 'Antimycin', 'antimycin a': 'Antimycin',
    '2-dg': '2-DG', '2dg': '2-DG',
    # Treatments agudos (Formato B)
    'acute dex': 'Acute-DEX', 'acute_dex': 'Acute-DEX', 'acute-dex': 'Acute-DEX',
    'dex-recovery': 'DEX-Recovery', 'dex recovery': 'DEX-Recovery',
    # Arrival / pre-study
    'arrival': 'Arrival', 'co...rrival': 'Arrival',
    # Seahorse measurement (no es tratamiento biológico, pero existe en imágenes)
    'seahorse': 'Seahorse-Measurement', 'seahorse well': 'Seahorse-Measurement',
    # Post-thaw
    'post-thaw': 'Post-Thaw', 'post thaw': 'Post-Thaw',
}

# Tokens que son ruido del nombre de archivo (NO son treatments)
NOISE_TOKENS = frozenset([
    'days', 'day', 'growth', 'kcells', 'cells', 'cell',
    'well', 'flask', 'passage', 'rep', 'replicate',
    'pre', 'study', 'prestudy', 'post', 'thaw', 'months',
    'of', 'for', 'with', 'and', 'or',
])

# Regex compilados
RE_CELL_LINE  = re.compile(r'^(hFB\d+(?:-[mf])?)', re.IGNORECASE)
RE_PASSAGE    = re.compile(r'(?:^|[\s_,])P(\d+)(?:[\s_,\.]|$)', re.IGNORECASE)
RE_TIMEPOINT  = re.compile(r'(?:^|[\s_])t(\d+)(?:[\s_]|$)', re.IGNORECASE)
RE_MAG_A      = re.compile(r'(?:^|[\s,])( \d{2}x)(?:[\s,\.]|$)', re.IGNORECASE)  # Formato A
RE_MAG_B      = re.compile(r'[_\s](\d{2}x)(?:[_\s]|$)', re.IGNORECASE)             # Formato B
RE_MAG_END    = re.compile(r'(\d{2}x)$', re.IGNORECASE)                             # al final
RE_NON_HFB    = re.compile(r'^(HEK|MCF|CHO|COS|NIH|HeLa|Vero)', re.IGNORECASE)    # no-fibroblastos


def normalize_treatment(raw: str) -> str:
    """Normaliza variantes de treatment a nombre canónico."""
    if not raw:
        return None
    key = raw.strip().lower()
    key = re.sub(r'[\s_]+', ' ', key).strip()
    if key in TREATMENT_CANON:
        return TREATMENT_CANON[key]
    # Si es solo tokens de ruido, no es un treatment válido
    tokens = set(re.split(r'[\s_\-,]+', key))
    clean_tokens = tokens - NOISE_TOKENS - {''}  
    if not clean_tokens:
        return None
    # Devolver capitalizado si no está en el dict pero parece válido
    return raw.strip().replace('_', ' ').title()


def get_magnif(stem: str, fmt: str) -> str | None:
    """Extrae magnificación tolerando ambos formatos de separador."""
    # Intentar con el regex del formato correcto primero
    if fmt == 'B':
        m = RE_MAG_B.search(stem)
        if m: return m.group(1).lower()
    # Fallback: buscar al final del string
    m = RE_MAG_END.search(stem)
    if m: return m.group(1).lower()
    # Fallback universal
    m = re.search(r'(\d{2}x)', stem, re.IGNORECASE)
    if m: return m.group(1).lower()
    return None


def detect_format(stem: str) -> str:
    """Detecta si el nombre usa espacio (A) o guión_bajo (B) como separador."""
    if RE_NON_HFB.match(stem):
        return 'SKIP'
    n_underscore = stem.count('_')
    n_space = stem.count(' ')
    # Si no empieza con hFB y tiene comas → posiblemente Formato C (sin cell_line)
    if not RE_CELL_LINE.match(stem) and ',' in stem:
        return 'C'
    return 'B' if n_underscore > n_space and n_underscore >= 2 else 'A'


def parse_format_A(stem: str) -> dict:
    """
    Formato A: 'hFB1-m P10  t175 4 days growth 10x, Ctrl'
    Treatment está después de la última coma, ANTES de la magnif si la hay.
    """
    info = {'img_cell_line': None, 'img_passage': None,
            'img_timepoint': None, 'img_magnif': None,
            'img_treatment': None, 'img_format': 'A', 'img_skip': False}

    m = RE_CELL_LINE.match(stem); info['img_cell_line'] = m.group(1) if m else None
    m = RE_PASSAGE.search(stem);  info['img_passage']   = int(m.group(1)) if m else None
    m = RE_TIMEPOINT.search(stem); info['img_timepoint'] = int(m.group(1)) if m else None
    info['img_magnif'] = get_magnif(stem, 'A')

    # Treatment: después de la ÚLTIMA coma del stem
    comma_parts = stem.rsplit(',', 1)
    if len(comma_parts) == 2:
        candidate = comma_parts[1].strip()
        # Descartar si el candidato ES la magnif (ej: ', 10x')
        if re.match(r'^\d{2}x$', candidate, re.IGNORECASE):
            candidate = None
        # Descartar si contiene magnif al final (ej: 'post-thaw, 10x')
        elif re.search(r',\s*\d{2}x$', candidate, re.IGNORECASE):
            candidate = re.sub(r',\s*\d{2}x$', '', candidate).strip()
        info['img_treatment'] = normalize_treatment(candidate) if candidate else None
    return info


def parse_format_B(stem: str) -> dict:
    """
    Formato B: 'hFB14_Acute_DEX_P27_7-days_250kcells_10x'
    Treatment = tokens que no son cell_line, passage, timepoint, magnif, ni ruido.
    """
    info = {'img_cell_line': None, 'img_passage': None,
            'img_timepoint': None, 'img_magnif': None,
            'img_treatment': None, 'img_format': 'B', 'img_skip': False}

    m = RE_CELL_LINE.match(stem); info['img_cell_line'] = m.group(1) if m else None
    m = RE_PASSAGE.search(stem);  info['img_passage']   = int(m.group(1)) if m else None
    m = RE_TIMEPOINT.search(stem); info['img_timepoint'] = int(m.group(1)) if m else None
    info['img_magnif'] = get_magnif(stem, 'B')

    # Treatment: separar por _ y filtrar tokens conocidos
    tokens = stem.split('_')
    treatment_tokens = []
    for tok in tokens:
        if RE_CELL_LINE.match(tok): continue           # es cell_line
        if re.match(r'^P\d+$', tok, re.IGNORECASE): continue  # es passage
        if re.match(r'^t\d+$', tok, re.IGNORECASE): continue  # es timepoint
        if re.match(r'^\d{2}x$', tok, re.IGNORECASE): continue  # es magnif
        if re.match(r'^\d+[km]?cells?$', tok, re.IGNORECASE): continue  # es conteo células
        if re.match(r'^\d+-?days?$', tok, re.IGNORECASE): continue  # son días
        if tok.lower() in NOISE_TOKENS: continue
        if tok.strip(): treatment_tokens.append(tok)

    if treatment_tokens:
        raw = ' '.join(treatment_tokens)
        info['img_treatment'] = normalize_treatment(raw)
    return info


def parse_format_C(stem: str, folder_cell_line: str = None) -> dict:
    """
    Formato C: 'ctrl, P10, 500kcells, t175, 5-days growth, 10x'
    No hay cell_line en el nombre → viene de la carpeta padre.
    """
    # Reusar el parser A para los campos que sí están
    info = parse_format_A(stem)
    info['img_format'] = 'C'
    # Usar cell_line de la carpeta si está disponible
    if info['img_cell_line'] is None and folder_cell_line:
        info['img_cell_line'] = folder_cell_line
    # El treatment en Formato C suele estar ANTES de la primera coma
    if info['img_treatment'] is None:
        first_part = stem.split(',')[0].strip()
        info['img_treatment'] = normalize_treatment(first_part)
    return info


def parse_image_name_v2(stem: str, folder_cell_line: str = None) -> dict:
    """
    Parser principal v2.1 — detecta formato y despacha al sub-parser correcto.
    
    Args:
        stem: nombre del archivo sin extensión
        folder_cell_line: cell_line extraída del nombre de carpeta (fallback)
    """
    fmt = detect_format(stem)
    if fmt == 'SKIP':
        return {'img_cell_line': None, 'img_passage': None, 'img_timepoint': None,
                'img_magnif': None, 'img_treatment': None, 'img_format': 'SKIP',
                'img_skip': True}
    elif fmt == 'B':
        return parse_format_B(stem)
    elif fmt == 'C':
        return parse_format_C(stem, folder_cell_line)
    else:
        return parse_format_A(stem)


print('✅ Parser v2.1 definido')
print(f'   Treatments canónicos: {len(TREATMENT_CANON)} variantes → {len(set(TREATMENT_CANON.values()))} categorías')

✅ Parser v2.1 definido
   Treatments canónicos: 41 variantes → 19 categorías


In [4]:
# ── Test del parser v2.1 ──────────────────────────────────────────────────────
test_cases = [
    # (stem, exp_cell_line, exp_passage, exp_magnif)
    # Formato A — baseline
    ('hFB1-m P10  t175 4 days growth 10x, Ctrl',        'hFB1-m',  10, '10x'),
    ('hFB1-m P10  t175 4 days growth 20x, MitoQ + DEX', 'hFB1-m',  10, '20x'),
    ('hFB1-m P10  t175 4 days growth 10x, NAC + DEX',   'hFB1-m',  10, '10x'),
    ('hFB14-m P3 t45 10x, Arrival',                      'hFB14-m',  3, '10x'),
    # Formato B — los que fallaban
    ('hFB14_Acute_DEX_P27_7-days_250kcells_10x',         'hFB14',   27, '10x'),
    ('hFB14_ctrl_P27_7-days_250kcells_10x',              'hFB14',   27, '10x'),
    ('hFB12_Acute-DEX_P26_7-days_250kcells_10x',         'hFB12',   26, '10x'),
    ('hFB12_DEX-Recovery_P26_7-days_250kcells_20x',      'hFB12',   26, '20x'),
    ('hFB12_DEX_P26_7-days_250kcells_20x',               'hFB12',   26, '20x'),
    ('hFB13_Acute-DEX_P32_7-days_250kcells_10x',         'hFB13',   32, '10x'),
    # Falsos positivos anteriores — ahora deben tener treatment=None o limpio
    ('seahorse well, 10x',                               None,    None, '10x'),
    ('3days growth, post-thaw, 10x',                     None,    None, '10x'),
    # SKIP
    ('HEK293, PXX, 250Kcells, t75, 7-days growth, 10x', None,    None, None),
    # Pre-study edge
    ('hFB14-m Co...rrival 10x',                          'hFB14-m', None, '10x'),
]

print(f'{"Stem (truncado)":<48} {"CL":<10} {"P":>4} {"Mag":>5} {"Treatment":<20} {"Fmt":>4} {"OK"}')
print('─' * 108)
n_pass = 0
for stem, exp_cl, exp_p, exp_mag in test_cases:
    r = parse_image_name_v2(stem)
    ok_cl  = r['img_cell_line'] == exp_cl
    ok_p   = r['img_passage']   == exp_p
    ok_mag = r['img_magnif']    == exp_mag
    ok = '✅' if (ok_cl and ok_p and ok_mag) else '❌'
    if ok == '✅': n_pass += 1
    print(f"{stem[:46]:<48} {str(r['img_cell_line']):<10} {str(r['img_passage']):>4} "
          f"{str(r['img_magnif']):>5} {str(r['img_treatment']):<20} {r['img_format']:>4} {ok}")

print(f'\n{n_pass}/{len(test_cases)} tests pasados')

Stem (truncado)                                  CL            P   Mag Treatment             Fmt OK
────────────────────────────────────────────────────────────────────────────────────────────────────────────
hFB1-m P10  t175 4 days growth 10x, Ctrl         hFB1-m       10   10x Control                 A ✅
hFB1-m P10  t175 4 days growth 20x, MitoQ + DE   hFB1-m       10   20x MitoQ+DEX               A ✅
hFB1-m P10  t175 4 days growth 10x, NAC + DEX    hFB1-m       10   10x NAC+DEX                 A ✅
hFB14-m P3 t45 10x, Arrival                      hFB14-m       3   10x Arrival                 A ✅
hFB14_Acute_DEX_P27_7-days_250kcells_10x         hFB14        27   10x Acute-DEX               B ✅
hFB14_ctrl_P27_7-days_250kcells_10x              hFB14        27   10x Control                 B ✅
hFB12_Acute-DEX_P26_7-days_250kcells_10x         hFB12        26   10x Acute-DEX               B ✅
hFB12_DEX-Recovery_P26_7-days_250kcells_20x      hFB12        26   20x DEX-Recovery            B ✅

## 4. Recorrido del árbol de imágenes con parser v2.1

In [5]:
print('📂 Recorriendo árbol de imágenes...')
IMG_EXTENSIONS = {'.jpg', '.jpeg', '.tif', '.tiff', '.png'}

# Regex para extraer cell_line de nombre de carpeta
RE_FOLDER_CELL = re.compile(r'(hFB\d+(?:-[mf])?)', re.IGNORECASE)

image_records = []
n_skip = 0

for img_path in sorted(IMAGES_ROOT.rglob('*')):
    if img_path.suffix.lower() not in IMG_EXTENSIONS:
        continue

    rel   = img_path.relative_to(IMAGES_ROOT)
    parts = rel.parts

    # Intentar extraer cell_line de los nombres de carpeta (fallback Formato C)
    folder_cell_line = None
    for part in parts[:-1]:  # todos menos el filename
        m = RE_FOLDER_CELL.search(part)
        if m:
            folder_cell_line = m.group(1)
            break

    parsed = parse_image_name_v2(img_path.stem, folder_cell_line=folder_cell_line)

    if parsed['img_skip']:
        n_skip += 1
        continue

    record = {
        'img_path':            str(img_path),
        'img_filename':        img_path.name,
        'img_stem':            img_path.stem,
        'img_ext':             img_path.suffix.lower(),
        'img_size_kb':         round(img_path.stat().st_size / 1e3, 1),
        'dir_depth':           len(parts) - 1,
        'phase_folder':        parts[0] if len(parts) > 1 else 'Unknown',
        'subfolder_1':         parts[1] if len(parts) > 2 else None,
        'subfolder_2':         parts[2] if len(parts) > 3 else None,
        'folder_cell_line':    folder_cell_line,
    }

    # Subfolder type
    if record['subfolder_1']:
        is_date = bool(re.match(r'\d{4}-\d{2}-\d{2}', record['subfolder_1']))
        is_pass = bool(re.match(r'^P\d+$', record['subfolder_1'], re.IGNORECASE))
        record['subfolder_1_type'] = 'date' if is_date else ('passage' if is_pass else 'other')
    else:
        record['subfolder_1_type'] = None

    record.update(parsed)
    image_records.append(record)

df_imgs = pd.DataFrame(image_records)

print(f'\n✅ {len(df_imgs)} imágenes indexadas  |  {n_skip} descartadas (HEK293 u otras no-fibroblastos)')
print(f'\n📊 Por Phase:')
print(df_imgs['phase_folder'].value_counts().to_string())
print(f'\n📊 Por formato detectado:')
print(df_imgs['img_format'].value_counts().to_string())
print(f'\n📊 Por magnificación:')
print(df_imgs['img_magnif'].value_counts().to_string())
print(f'\n📊 Cell lines detectadas:')
print(df_imgs['img_cell_line'].value_counts().to_string())
print(f'\n📊 Treatments normalizados:')
print(df_imgs['img_treatment'].value_counts().to_string())

📂 Recorriendo árbol de imágenes...

✅ 3265 imágenes indexadas  |  2 descartadas (HEK293 u otras no-fibroblastos)

📊 Por Phase:
phase_folder
PhaseII     1553
PhaseV      1085
PhaseI       425
PhaseIII     202

📊 Por formato detectado:
img_format
A    2199
B    1018
C      48

📊 Por magnificación:
img_magnif
10x    1431
20x    1379

📊 Cell lines detectadas:
img_cell_line
hFB12      532
hFB13      490
hFB14      282
hFB1-m     280
hFB2-f     148
hFB7       110
hFB6        97
hFB8        86
hFB11       77
hFB10       11
hFB5         9
hFB9         6
hFB4-f       2
hFB14-m      2

📊 Treatments normalizados:
img_treatment
Post-Treatment 2                              156
Control                                        50
DEX                                            50
MitoQ                                          48
NAC                                            48
NAC+DEX                                        47
Alpha-Keto                                     47
MitoQ+DEX                 

In [6]:
# ── Diagnóstico comparativo antes/después ──────────────────────────────────────
parse_failures = df_imgs[
    df_imgs['img_cell_line'].isna() |
    df_imgs['img_passage'].isna()   |
    df_imgs['img_magnif'].isna()
].copy()

pct_fail = len(parse_failures) / len(df_imgs) * 100
print(f'⚠️  Parsing incompleto: {len(parse_failures)} / {len(df_imgs)} ({pct_fail:.1f}%)')
print(f'   (v2.0 tenía: 1188 / 3267 = 36.4%)')
print(f'   Mejora: {36.4 - pct_fail:.1f} pp')

if len(parse_failures) > 0:
    print(f'\nDesglose de fallos restantes:')
    print(f'  sin cell_line: {df_imgs["img_cell_line"].isna().sum()}')
    print(f'  sin passage:   {df_imgs["img_passage"].isna().sum()}')
    print(f'  sin magnif:    {df_imgs["img_magnif"].isna().sum()}')
    print(f'\nEjemplos de fallos restantes (primeros 15):')
    for _, row in parse_failures.head(15).iterrows():
        print(f'  [{row["phase_folder"]}/{row["subfolder_1"]}]  {row["img_filename"]}')
        print(f'    → fmt={row["img_format"]}  cl={row["img_cell_line"]}  '
              f'p={row["img_passage"]}  mag={row["img_magnif"]}  t={row["img_treatment"]}')

# Guardar catálogo actualizado
imgs_catalog_path = OUTPUT_DIR / 'image_catalog_v2.csv'
df_imgs.to_csv(imgs_catalog_path, index=False)
print(f'\n💾 Catálogo guardado: {imgs_catalog_path}')

⚠️  Parsing incompleto: 1166 / 3265 (35.7%)
   (v2.0 tenía: 1188 / 3267 = 36.4%)
   Mejora: 0.7 pp

Desglose de fallos restantes:
  sin cell_line: 1133
  sin passage:   485
  sin magnif:    455

Ejemplos de fallos restantes (primeros 15):
  [PhaseII/other_treatments]  T80, hFB1-m, P5, 7-days Growth, 500kcells, 10x.jpg
    → fmt=C  cl=nan  p=5.0  mag=10x  t=T80
  [PhaseII/other_treatments]  T80, hFB1-m, P5, 7-days Growth, 500kcells, 20x.jpg
    → fmt=C  cl=nan  p=5.0  mag=20x  t=T80
  [PhaseII/other_treatments]  T80, hFB1-m, P6, 7-days Growth, 250kcells, 10x.jpg
    → fmt=C  cl=nan  p=6.0  mag=10x  t=T80
  [PhaseII/other_treatments]  T80, hFB1-m, P6, 7-days Growth, 250kcells, 20x.jpg
    → fmt=C  cl=nan  p=6.0  mag=20x  t=T80
  [PhaseII/other_treatments]  T80, hFB1-m, P7, 6-days Growth, 250kcells, 10x.jpg
    → fmt=C  cl=nan  p=7.0  mag=10x  t=T80
  [PhaseII/other_treatments]  T80, hFB1-m, P7, 6-days Growth, 250kcells, 20x.jpg
    → fmt=C  cl=nan  p=7.0  mag=20x  t=T80
  [PhaseII/other_

## 5. Normalización del join: resolver `hFB14` vs `hFB14-m`

El Formato B da `hFB14` (sin sufijo `-m`/`-f`) pero el CSV tiene `hFB14-m`.  
La solución: normalizar ambos a la **clave numérica** `hfb14` para el join, y guardar el valor original en otra columna.

In [ ]:
def cell_line_join_key(raw: str) -> str | None:
    """
    Clave de join normalizada.
    hFB1-m → hfb1   hFB14-m → hfb14   hFB12 → hfb12
    Elimina el sufijo -m/-f para que hFB14 y hFB14-m casen.
    """
    if pd.isna(raw) or raw is None:
        return None
    s = str(raw).strip().lower()
    s = re.sub(r'\s+', '', s)       # quitar espacios
    s = re.sub(r'-[mf]$', '', s)    # quitar sufijo -m o -f
    return s  # ej: 'hfb14', 'hfb1', 'hfb12'

# Aplicar a CSV
df['cell_line_key'] = df['cell_line'].apply(cell_line_join_key)

# Aplicar a imágenes
df_imgs['img_cell_line_key'] = df_imgs['img_cell_line'].apply(cell_line_join_key)

# Normalizar treatment del CSV también
def normalize_treatment_csv(raw):
    if pd.isna(raw): return 'Unknown'
    key = str(raw).strip().lower()
    key = re.sub(r'[\s_]+', ' ', key)
    return TREATMENT_CANON.get(key, str(raw).strip())

df['treatment_key'] = df['treatment'].apply(normalize_treatment_csv)
df_imgs['img_treatment_key'] = df_imgs['img_treatment'].fillna('Unknown')

print('Claves de join en CSV (cell_line_key):')
print(df['cell_line_key'].value_counts().to_string())
print('\nClaves de join en imágenes (img_cell_line_key):')
print(df_imgs['img_cell_line_key'].value_counts().to_string())

# Verificar solapamiento
csv_keys = set(df['cell_line_key'].dropna())
img_keys = set(df_imgs['img_cell_line_key'].dropna())
print(f'\n🔑 Overlap CSV ↔ imágenes: {csv_keys & img_keys}')
print(f'   Solo en CSV:   {csv_keys - img_keys}')
print(f'   Solo en imgs:  {img_keys - csv_keys}')

## 6. Join CSV ↔ Imágenes

In [ ]:
print('🔗 Construyendo índice de imágenes...')

# Índice: (cell_line_key, passage, treatment_key) → lista de paths
img_index_exact   = defaultdict(list)  # match exacto
img_index_partial = defaultdict(list)  # match sin treatment

for _, row in df_imgs.iterrows():
    cl = row['img_cell_line_key']
    p  = row['img_passage']
    t  = row['img_treatment_key']
    if cl is None: continue
    if p is not None:  # solo indexar si tenemos passage
        img_index_exact[(cl, p, t)].append(row['img_path'])
        img_index_partial[(cl, p)].append(row['img_path'])

print(f'   Llaves exactas en índice:   {len(img_index_exact)}')
print(f'   Llaves parciales en índice: {len(img_index_partial)}')

def find_images(row):
    cl = row.get('cell_line_key')
    p  = row.get('passage')
    t  = row.get('treatment_key', 'Unknown')
    if cl is None or p is None:
        return []
    # 1. Match exacto: cell_line + passage + treatment
    key = (cl, p, t)
    if key in img_index_exact:
        return img_index_exact[key]
    # 2. Fallback: cell_line + passage (cualquier treatment)
    key2 = (cl, p)
    if key2 in img_index_partial:
        return img_index_partial[key2]
    return []

df['matched_images']  = df.apply(find_images, axis=1)
df['n_images_10x']    = df['matched_images'].apply(lambda imgs: sum(1 for p in imgs if '10x' in Path(p).stem.lower()))
df['n_images_20x']    = df['matched_images'].apply(lambda imgs: sum(1 for p in imgs if '20x' in Path(p).stem.lower()))
df['n_images_total']  = df['matched_images'].apply(len)
df['has_image']       = df['n_images_total'] > 0

print(f'\n📸 Resultado del join:')
print(f'   Muestras con ≥1 imagen: {df["has_image"].sum()} / {len(df)} ({df["has_image"].mean()*100:.1f}%)')
print(f'   Muestras sin imagen:    {(~df["has_image"]).sum()}')
print(f'   Total imágenes linkadas: {df["n_images_total"].sum()}')
print(f'   Media imágenes/muestra (con imagen): {df.loc[df["has_image"],"n_images_total"].mean():.1f}')

In [ ]:
# ── Diagnóstico: muestras sin imagen ──────────────────────────────────────────
no_img = df[~df['has_image']]
diag_cols = [c for c in ['sample_id','cell_line_key','passage','treatment_key','pdl_bin'] if c in df.columns]
print(f'⚠️  {len(no_img)} muestras sin imagen')
print('\nPor cell_line:')
print(no_img['cell_line_key'].value_counts().to_string())
print('\nPor pdl_bin:')
print(no_img['pdl_bin'].value_counts().to_string() if 'pdl_bin' in no_img else 'N/A')

## 7. Flags de modalidades, manifest final y splits

In [ ]:
# ── Flags de disponibilidad por modalidad ─────────────────────────────────────
df['has_rna']             = df['rnaseq_id'].notna()              if 'rnaseq_id' in df.columns else False
df['has_methylation']     = df['methylation_basename'].notna()   if 'methylation_basename' in df.columns else False
df['has_telomere']        = df['telomere_length'].notna()        if 'telomere_length' in df.columns else False
df['has_mtdna']           = df['mtdna_cn'].notna()               if 'mtdna_cn' in df.columns else False
df['has_seahorse']        = df['seahorse_basal'].notna()         if 'seahorse_basal' in df.columns else False
df['has_epigenetic_clocks'] = df['clock_horvath1'].notna()       if 'clock_horvath1' in df.columns else False

modality_flags = [
    ('has_image',            '📸 Imágenes brightfield'),
    ('has_rna',              '🧬 RNA-seq bulk'),
    ('has_methylation',      '🔬 Metilación EPIC'),
    ('has_telomere',         '📍 Telómero (qPCR)'),
    ('has_mtdna',            '🔋 mtDNA copy number'),
    ('has_seahorse',         '⚡ Bioenergética (Seahorse)'),
    ('has_epigenetic_clocks','🕰️  Relojes epigenéticos'),
]

flag_cols = [f for f, _ in modality_flags if f in df.columns]
df['n_modalities'] = df[flag_cols].sum(axis=1)
df['modalities_str'] = df[flag_cols].apply(
    lambda r: '|'.join(f.replace('has_','') for f in flag_cols if r[f]), axis=1
)

print('=' * 62)
print('DISPONIBILIDAD POR MODALIDAD')
print('=' * 62)
for flag, label in modality_flags:
    if flag in df.columns:
        n = int(df[flag].sum())
        pct = n / len(df) * 100
        bar = '█' * int(pct/5) + '░' * (20 - int(pct/5))
        print(f'  {label:<35s} {bar} {n:5d}/{len(df)} ({pct:.0f}%)')

print(f'\nMuestras con ≥2 modalidades: {(df["n_modalities"] >= 2).sum()}')
print(f'Muestras con ≥3 modalidades: {(df["n_modalities"] >= 3).sum()}')

In [ ]:
# ── Manifest final ────────────────────────────────────────────────────────────
from sklearn.model_selection import GroupKFold

MANIFEST_COLS = [
    'sample_id','cell_line','cell_line_key','cell_line_group',
    'clinical_condition','sex','donor_age',
    'passage','timepoint','pdl','pdl_norm','pdl_bin',
    'pct_lifespan','days_grown','date_passage',
    'treatment','treatment_key','treatment_desc',
    'study_part','tech_rep','freeze_thaw_count',
    'has_image','n_images_total','n_images_10x','n_images_20x',
    'has_rna','rnaseq_id','has_methylation','methylation_basename',
    'has_telomere','has_mtdna','has_seahorse','has_epigenetic_clocks',
    'n_modalities','modalities_str',
    'telomere_length','mtdna_cn','cell_size','pct_dead','il6','gdf15',
    'seahorse_basal','seahorse_max','seahorse_atp','seahorse_coupling','atp_ratio',
    'clock_horvath1','clock_phenoage','clock_grimage',
    'clock_mitotic','clock_dnam_telomere','clock_dnam_sen','clock_dunedin',
    'matched_images',
]

manifest_cols_present = [c for c in MANIFEST_COLS if c in df.columns]
df_manifest = df[manifest_cols_present].copy()
df_manifest['matched_images'] = df_manifest['matched_images'].apply(
    lambda x: '|'.join(x) if isinstance(x, list) else ''
)

# GroupKFold por cell_line
groups = df_manifest['cell_line_key'].fillna('unknown')
n_splits = min(5, groups.nunique())
gkf = GroupKFold(n_splits=n_splits)
df_manifest['split_fold'] = -1
y_dummy = df_manifest['pdl'].fillna(0)
for fold_idx, (train_idx, val_idx) in enumerate(gkf.split(df_manifest, y_dummy, groups)):
    df_manifest.iloc[val_idx, df_manifest.columns.get_loc('split_fold')] = fold_idx

print('✅ Splits GroupKFold por cell_line:')
for fold in range(n_splits):
    fd = df_manifest[df_manifest['split_fold'] == fold]
    print(f'  Fold {fold}: {len(fd):4d} muestras | {list(fd["cell_line_key"].unique())}')

print(f'\nManifest final: {df_manifest.shape[0]} × {df_manifest.shape[1]}')

## 8. Guardar + Reporte QC visual

In [ ]:
def md5_file(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''): h.update(chunk)
    return h.hexdigest()

timestamp = datetime.now().strftime('%Y%m%d_%H%M')
manifest_path   = OUTPUT_DIR / f'sturm_manifest_v{PIPELINE_VERSION}_{timestamp}.csv'
manifest_latest = OUTPUT_DIR / 'manifest_latest.csv'

df_manifest.to_csv(manifest_path, index=False)
df_manifest.to_csv(manifest_latest, index=False)
checksum = md5_file(manifest_path)

# Reporte JSON
report = {
    'pipeline_version': PIPELINE_VERSION,
    'run_date': RUN_DATE,
    'manifest_file': manifest_path.name,
    'manifest_md5': checksum,
    'n_samples': len(df_manifest),
    'n_cell_lines': int(df_manifest['cell_line_key'].nunique()),
    'pdl_range': [float(df_manifest['pdl'].min()), float(df_manifest['pdl'].max())],
    'parser_stats': {
        'total_images_indexed': len(df_imgs),
        'images_skipped_non_fb': int(n_skip),
        'format_A': int((df_imgs['img_format'] == 'A').sum()),
        'format_B': int((df_imgs['img_format'] == 'B').sum()),
        'format_C': int((df_imgs['img_format'] == 'C').sum()),
        'parse_failures': int(parse_failures.shape[0]),
        'parse_failure_pct': round(len(parse_failures)/len(df_imgs)*100, 1),
    },
    'modality_coverage': {f: int(df_manifest[f].sum()) for f, _ in modality_flags if f in df_manifest},
    'multimodal_samples_2plus': int((df_manifest['n_modalities'] >= 2).sum()),
    'multimodal_samples_3plus': int((df_manifest['n_modalities'] >= 3).sum()),
    'split_folds': n_splits,
}

report_path = OUTPUT_DIR / 'logs' / f'pipeline_report_{timestamp}.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print('📋 REPORTE FINAL')
print(json.dumps(report, indent=2))

In [ ]:
# ── QC Visual ─────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 12))
fig.suptitle(f'Sturm Dataset — Manifest QC | v{PIPELINE_VERSION} | {RUN_DATE}',
             fontsize=13, fontweight='bold')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# 1. Disponibilidad por modalidad
ax1 = fig.add_subplot(gs[0, 0])
flag_labels_present = [(f, l) for f, l in modality_flags if f in df_manifest.columns]
counts = [int(df_manifest[f].sum()) for f, _ in flag_labels_present]
labels_s = [l.split(' ', 1)[1] for _, l in flag_labels_present]
ax1.barh(labels_s, counts, color='#4e79a7', edgecolor='white')
ax1.set_xlabel('N muestras'); ax1.set_title('Disponibilidad por modalidad')
for i, c in enumerate(counts): ax1.text(c+5, i, str(c), va='center', fontsize=8)

# 2. Cobertura multimodal
ax2 = fig.add_subplot(gs[0, 1])
mod_dist = df_manifest['n_modalities'].value_counts().sort_index()
ax2.bar(mod_dist.index.astype(str), mod_dist.values, color='#59a14f', edgecolor='white')
ax2.set_xlabel('N modalidades'); ax2.set_ylabel('N muestras')
ax2.set_title('Cobertura multimodal por muestra')

# 3. Treatments normalizados (top 10)
ax3 = fig.add_subplot(gs[0, 2])
if 'treatment_key' in df_manifest.columns:
    top_treats = df_manifest['treatment_key'].value_counts().head(10)
    ax3.barh(top_treats.index[::-1], top_treats.values[::-1], color='#f28e2b', edgecolor='white')
    ax3.set_xlabel('N muestras'); ax3.set_title('Top treatments (normalizados)')

# 4. PDL por donante
ax4 = fig.add_subplot(gs[1, :])
if 'pdl' in df_manifest.columns and 'cell_line_key' in df_manifest.columns:
    cell_lines = sorted(df_manifest['cell_line_key'].dropna().unique())
    cmap = plt.cm.get_cmap('tab10', len(cell_lines))
    bin_colors = {'early': '#59a14f', 'mid': '#f28e2b', 'late': '#e15759'}
    for i, cl in enumerate(cell_lines):
        sub = df_manifest[(df_manifest['cell_line_key']==cl) & df_manifest['pdl'].notna()]
        if len(sub) == 0: continue
        for bin_name, bc in bin_colors.items():
            b = sub[sub['pdl_bin']==bin_name] if 'pdl_bin' in sub else sub
            ax4.scatter(b['pdl'], [i]*len(b), s=18, color=bc, alpha=0.5, zorder=3)
    ax4.set_yticks(range(len(cell_lines)))
    ax4.set_yticklabels(cell_lines, fontsize=9)
    ax4.set_xlabel('PDL')
    ax4.set_title('Cobertura de PDL por cell_line  (🟢early  🟠mid  🔴late)')
    ax4.grid(axis='x', alpha=0.3)

plt.savefig(OUTPUT_DIR / 'reports' / f'manifest_qc_v{PIPELINE_VERSION}_{timestamp}.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('✅ Reporte QC guardado')

## 9. Subconjunto MVP-1 (encoder visual)

In [ ]:
mvp1_mask = df_manifest['has_image'] & df_manifest['pdl'].notna()
df_mvp1   = df_manifest[mvp1_mask].copy()
df_mvp1_ctrl = df_mvp1[df_mvp1['treatment_key'] == 'Control'] if 'treatment_key' in df_mvp1 else df_mvp1

print('=' * 55)
print('SUBCONJUNTO MVP-1 (encoder visual)')
print('=' * 55)
print(f'  Total muestras con imagen + PDL:  {len(df_mvp1)}')
print(f'  Solo condición Control:           {len(df_mvp1_ctrl)}')
print(f'  Imágenes 10x (ctrl):              {df_mvp1_ctrl["n_images_10x"].sum()}')
print(f'  Imágenes 20x (ctrl):              {df_mvp1_ctrl["n_images_20x"].sum()}')
print(f'  Cell lines representadas:         {df_mvp1_ctrl["cell_line_key"].nunique()}')
print(f'  Rango PDL (ctrl):                 {df_mvp1_ctrl["pdl"].min():.1f} → {df_mvp1_ctrl["pdl"].max():.1f}')
print(f'  Distribución pdl_bin (ctrl):')
if 'pdl_bin' in df_mvp1_ctrl: print(df_mvp1_ctrl['pdl_bin'].value_counts().to_string())

mvp1_path = OUTPUT_DIR / f'mvp1_manifest_{timestamp}.csv'
df_mvp1.to_csv(mvp1_path, index=False)
print(f'\n💾 MVP-1 guardado: {mvp1_path}')
print('\n✅ Pipeline v2.1 completo.')